注意：运行下面的代码首先要执行 [gen_demo_factor_data.py](../tools/gen_demo_factor_data.py) 脚本生成示例数据。

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import logging

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

In [ ]:
import datetime as dt

import numpy as np
import pandas as pd


from QuantStudio.Tools.Visualization import qs_help
from QuantStudio.Factor.HDF5DB import HDF5DB

FDB = HDF5DB(args={"MainDir": "../data/HDF5"}).connect()

# 因子定义框架

因子框架的整体架构（三层数据模型、基础因子/衍生因子的概念、与计算图的关系）请参见 **[基本框架](基本框架.ipynb)**，这里不再赘述。本文档聚焦于因子运算的具体使用。

## 因子分类回顾

* **基础因子**：通过 `FactorTable.getFactor()` 获取，或通过 `DataFactor` 直接构造
* **衍生因子**：由算子（`FactorOperator`）作用于描述子产生，按运算类型分为四类：

| 算子类 | 衍生因子类 | 依赖范围 |
|--------|------------|----------|
| `PointOperator` | `PointOperation` | 同时点、同证券 |
| `TimeOperator` | `TimeOperation` | 历史时序、同证券 |
| `SectionOperator` | `SectionOperation` | 同时点、全截面 |
| `PanelOperator` | `PanelOperation` | 历史时序 + 全截面 |

以 HSIGMA 因子的定义举例说明。HSIGMA 因子是 Barra 中国市场风险模型（CNE5）里的历史残余波动率因子，其构造过程如下：
* 首先由股票收盘价和昨收盘价因子计算收益率因子，这是**单点运算**；
* 然后用股票的收益率因子和市场收益率因子进行时间序列回归得到残余收益率因子 EPSILON，这是**时序运算**；
* 最后取一段时间的残余收益率求其标准差得到残余波动率因子 HSIGMA，这也是**时序运算**。

```mermaid
graph BT
    A(("昨收盘价<br/>(基础因子)")) --> B(("日收益率<br/>(单点运算)"))
    C(("收盘价<br/>(基础因子)")) --> B
    B --> D(("BETA<br/>(时序运算)"))
    E(("无风险利率<br/>(基础因子)")) --> D
    F(("市场收益<br/>(基础因子)")) --> D
    D --> G(("EPSILON<br/>(时序运算)"))
    B --> G
    F --> G
    G --> H(("HSIGMA<br/>(时序运算)"))
```

# 因子算子

## 算子基类

所有因子算子均继承自 `FactorOperator`（`QuantStudio.Factor.FactorOperation.FactorOperator`）。其核心参数：

| 参数 | 类型 | 说明 |
|------|------|------|
| `OperatorType` | `"Point" \| "Time" \| "Section" \| "Panel"` | 算子类型（冻结） |
| `Arity` | `int \| None` | 入参数量，即描述子个数；`None` 表示不限制 |
| `DataType` | `"double" \| "string" \| "object"` | 算子输出数据类型 |
| `ModelArgs` | `dict` | 传递给 `calculate` 的附加参数 |
| `MultiMapping` | `bool` | 是否多重映射（输出数据在某时点某 ID 处有多个值） |
| `CompoundType` | `list` | 复合输出类型（输出为多个子字段的复合结构） |

每个具体的算子必须实现 `calculate` 方法，该方法构成了算子的核心运算逻辑。

In [ ]:
from QuantStudio.Factor.FactorOperation import FactorOperator

print(qs_help(FactorOperator))

In [ ]:
print(qs_help(FactorOperator.calculate))

## `calculate` 方法的参数说明

```python
def calculate(self, f: Factor, idt, iid, x: list, args: dict) -> Any
```

| 参数 | 说明 |
|------|------|
| `f` | 该算子所属的因子对象 |
| `idt` | 当前待计算的时点，`DTMode="单时点"` 时为单个 `datetime`，`"多时点"` 时为 `list[datetime]` |
| `iid` | 当前待计算的 ID，`IDMode="单ID"` 时为单个 `str`，`"多ID"` 时为 `list[str]`（注意并发时 iid 不一定是全截面） |
| `x` | 描述子当期数据列表，元素格式取决于算子类型和 DTMode/IDMode |
| `args` | 来自 `ModelArgs` 的附加参数字典 |

## 创建自定义算子

有三种方式创建自定义算子：

1. **工厂函数 `makeFactorOperator`**：最常用，将函数转换为算子对象
2. **装饰器 `FactorOperatorized`**：语法糖，在函数定义时直接转换
3. **子类化**：直接继承 `PointOperator`/`TimeOperator`/`SectionOperator`/`PanelOperator` 并实现 `calculate`。较为繁琐，尽量避免

算子对象实现了 `__call__` 方法，作用在描述子上即可创建新的衍生因子。

In [ ]:
# makeFactorOperator 算子工厂函数
from QuantStudio.Factor.FactorOperation import makeFactorOperator

print(qs_help(makeFactorOperator))

In [ ]:
# 工厂函数创建算子
from QuantStudio.Factor.FactorOperation import makeFactorOperator

def calcMid(f, idt, iid, x, args):
    """计算两个因子的中间值"""
    return (x[0] + x[1]) / 2

calcMid = makeFactorOperator(func=calcMid, operator_type="Point", args={"Name": "calcMid", "Arity": 2})
print(qs_help(calcMid))

In [ ]:
# FactorOperatorized 装饰器
from QuantStudio.Factor.FactorOperation import FactorOperatorized

@FactorOperatorized(operator_type="Point", args={"Name": "calcMid", "Arity": 2})
def calcMid(f, idt, iid, x, args):
    """计算两个因子的中间值"""
    return (x[0] + x[1]) / 2

print(qs_help(calcMid))

In [ ]:
# 通过算子创建衍生因子
FT = FDB.getTable("stock_cn_day_bar")
High, Low = FT.getFactor("high"), FT.getFactor("low")

Mid = calcMid(High, Low, factor_args={"Name": "Mid"})
print(qs_help(Mid))

## 衍生因子的基类：`DerivativeFactor`

所有通过算子生成的衍生因子都继承自 `DerivativeFactor(Factor)`，其核心参数：

| 参数 | 说明 |
|------|------|
| `Operator` | 创建该因子的算子对象（冻结） |
| `ModelArgs` | 传递给算子 `calculate` 方法的附加参数字典 |

衍生因子具有属性 `Operator`，即构造它的算子对象。因子的元信息（如 `DataType`）从算子获取。

`DerivativeFactor` 的四个子类对应于四类运算：`PointOperation`、`TimeOperation`、`SectionOperation`、`PanelOperation`。直接实例化这些子类（传入 `descriptors` 列表和包含 `Operator` 的 `args`）也是一种构造衍生因子的方式，但通常更推荐通过算子的 `__call__` 方法（即 `Operator(factor)`）来创建。

## `readData` 的额外参数

因子（包括衍生因子）的 `readData` 方法支持以下 `kwargs` 来控制计算行为：

| 参数 | 说明 |
|------|------|
| `dt_ruler` | 时点标尺序列，默认为 `dts`。对时序/面板运算有实际影响，决定了回溯数据的可用范围 |
| `section_ids` | 截面 ID 序列。对截面/面板运算有实际影响，决定了运算时的全体截面范围；若不指定，则默认为因子参数中的 `SectionIDs` 或 `ids` |

# 单点运算

固定时点、固定 ID 的不同因子间的运算。是最简单的一类因子运算。典型例子是估值因子（PB、PE 等）。

## 核心参数

`PointOperator` 继承了 `FactorOperator` 的全部参数，并新增：

| 参数 | 取值 | 说明 |
|------|------|------|
| `DTMode` | `"单时点"`（默认）/ `"多时点"` | 每次调用 `calculate` 处理的时点数量 |
| `IDMode` | `"单ID"`（默认）/ `"多ID"` | 每次调用 `calculate` 处理的 ID 数量 |

`DTMode` 和 `IDMode` 的组合影响 `calculate` 函数接收到的 `x` 参数形状：选择 `"多时点"` + `"多ID"` 效率最高，选择 `"单时点"` + `"单ID"` 最灵活。

In [ ]:
from QuantStudio.Factor.FactorOperation import PointOperator
print(qs_help(PointOperator.calculate))

下面演示四种 `DTMode` / `IDMode` 组合下 `calculate` 的入参差异。

In [ ]:
# DTMode="多时点", IDMode="多ID"：一次处理全部数据，最高效
FT = FDB.getTable(table_name="stock_cn_day_bar")
High, Low = FT.getFactor("high"), FT.getFactor("low")

@FactorOperatorized(operator_type="Point", args={"Name": "calcMid", "Arity": 2, "DataType": "double", "DTMode": "多时点", "IDMode": "多ID"})
def PointFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print("-" * 10)
    return (x[0] + x[1]) / 2

PointFactor = PointFun(High, Low)
IDs = ["000001.SZ", "000002.SZ"]
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 3))
print(PointFactor.readData(ids=IDs, dts=DTs))

In [ ]:
# DTMode="单时点", IDMode="多ID"：按每个时点分别调用
@FactorOperatorized(operator_type="Point", args={"Name": "calcMid", "Arity": 2, "DataType": "double", "DTMode": "单时点", "IDMode": "多ID"})
def PointFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print("-" * 10)
    return (x[0] + x[1]) / 2

PointFactor = PointFun(High, Low)
print(PointFactor.readData(ids=IDs, dts=DTs))

In [ ]:
# DTMode="多时点", IDMode="单ID"：按每个 ID 分别调用
@FactorOperatorized(operator_type="Point", args={"Name": "calcMid", "Arity": 2, "DataType": "double", "DTMode": "多时点", "IDMode": "单ID"})
def PointFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print("-" * 10)
    return (x[0] + x[1]) / 2

PointFactor = PointFun(High, Low)
print(PointFactor.readData(ids=IDs, dts=DTs))

In [ ]:
# DTMode="单时点", IDMode="单ID"：最细粒度，每个时点-每个 ID 分别调用
@FactorOperatorized(operator_type="Point", args={"Name": "calcMid", "Arity": 2, "DataType": "double", "DTMode": "单时点", "IDMode": "单ID"})
def PointFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print("-" * 10)
    return (x[0] + x[1]) / 2

PointFactor = PointFun(High, Low)
print(PointFactor.readData(ids=IDs, dts=DTs))

# 时序运算

固定 ID 不同因子间在时间序列上的运算。除了指定描述子和定义算子外，还需指定每个描述子的**回溯期数**。典型例子是移动平均线。

## 核心参数

`TimeOperator` 继承了 `FactorOperator` 的全部参数，并新增以下与时序相关的核心参数：

| 参数 | 类型 | 说明 |
|------|------|------|
| `DTMode` | `"单时点"` / `"多时点"` | 每次调用处理的时点数量 |
| `IDMode` | `"单ID"` / `"多ID"` | 每次调用处理的 ID 数量 |
| `LookBack` | `list[int]` | 每个描述子向前回溯的时点数（不包括当前时点）。长度必须与 `Arity` 一致 |
| `StartDT` | `list[datetime \| None]` | 每个描述子的扩张窗口起始时点。`None` 表示使用 `LookBack` 的滚动窗口模式；非 `None` 表示从该时点开始取数据的扩张窗口模式 |
| `iInitFactor` | `int` | 自身迭代因子的索引。`>=0` 表示该索引的描述子是自身的上一期值（用于递归定义如 EMA）；`-1` 表示无自身迭代 |

In [ ]:
from QuantStudio.Factor.FactorOperation import TimeOperator
print(qs_help(TimeOperator.calculate))

## 回溯模式

时序运算支持两种回溯模式，由 `LookBack` 和 `StartDT` 的组合决定：

| 模式 | 参数设置 | 行为 |
|------|----------|------|
| **滚动窗口** | `StartDT[i]=None` | 只取最近 `LookBack[i] + 1` 个时点的数据，窗口随时间滑动 |
| **扩张窗口** | `StartDT[i]!=None` | 从 `StartDT[i]` 开始的所有历史数据，窗口不断扩大 |

In [ ]:
# 滚动窗口模式（无自身迭代）：LookBack=[2], StartDT=[None], iInitFactor=-1
FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

@FactorOperatorized(operator_type="Time", args={"Name": "calcMA", "Arity": 1, "DataType": "double", "LookBack": [3-1], "StartDT": [None], "iInitFactor": -1, "DTMode": "多时点", "IDMode": "多ID"})
def TimeFun(f, idt, iid, x, args):
    print(f"idt : {idt}")  # 注意 idt 包含了回溯的额外时点
    print(f"iid : {iid}")
    print(f"x[0] : {x[0]}")
    print("-" * 10)
    w = f.Operator.Args["LookBack"][0]
    return pd.DataFrame(x[0]).rolling(window=w+1).mean().values[w:]

TimeFactor = TimeFun(Close, factor_args={"Name": "MA"})

DTRuler = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 10))
DTs = DTRuler[-3:]
print(TimeFactor.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs, dt_ruler=DTRuler))

In [ ]:
# 扩张窗口模式（无自身迭代）：LookBack=[0], StartDT=[dt.datetime(2025,1,5)], iInitFactor=-1
@FactorOperatorized(operator_type="Time", args={"Name": "calcMA", "Arity": 1, "DataType": "double", "LookBack": [1-1], "StartDT": [dt.datetime(2025, 1, 5)], "iInitFactor": -1, "DTMode": "单时点", "IDMode": "多ID"})
def TimeFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    print(f"x[0] : {x[0]}")
    print("-" * 10)
    return pd.DataFrame(x[0]).mean().values

TimeFactor = TimeFun(Close, factor_args={"Name": "MA"})
print(TimeFactor.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs, dt_ruler=DTRuler))

In [ ]:
# 自身迭代 + 滚动窗口（如 EMA）：iInitFactor=0, StartDT=[None, None]
# 注意：自身迭代 + 滚动窗口模式下系统会强制关闭缓存，因为不同缓存状态下数据会不一致
@FactorOperatorized(operator_type="Time", args={"Name": "calcEMA", "Arity": 2, "DataType": "double", "LookBack": [2-1, 1-1], "StartDT": [None, None], "iInitFactor": 0, "DTMode": "单时点", "IDMode": "多ID"})
def TimeFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print("-" * 10)
    return 0.5 * x[0][0] + 0.5 * x[1][0]

TimeFactor = TimeFun(1, Close, factor_args={"Name": "EMA"})
print(TimeFactor.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs, dt_ruler=DTRuler))

In [ ]:
# 自身迭代 + 扩张窗口
@FactorOperatorized(operator_type="Time", args={"Name": "TimeFun", "Arity": 2, "DataType": "double", "LookBack": [2-1, 1-1], "StartDT": [dt.datetime(2025, 1, 5), dt.datetime(2025, 1, 5)], "iInitFactor": 0, "DTMode": "单时点", "IDMode": "多ID"})
def TimeFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    for i, ix in enumerate(x):
        print(f"x[{i}] : {ix}")
    print("-" * 10)
    return 0.5 * np.nanmean(x[0][:-1], axis=0) + 0.5 * np.nanmean(x[1], axis=0)

TimeFactor = TimeFun(1, Close)
print(TimeFactor.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs, dt_ruler=DTRuler))

# 截面运算

固定时点不同因子间在横截面上的运算。典型例子是数据标准化（Z-score）。

## 核心参数

`SectionOperator` 继承了 `FactorOperator` 的全部参数，并新增：

| 参数 | 类型 | 说明 |
|------|------|------|
| `DTMode` | `"单时点"` / `"多时点"` | 每次调用处理的时点数量 |
| `DescriptorSection` | `list[list[str] \| None]` | 每个描述子的截面范围。`None` 表示与当前因子的截面一致；指定 ID 列表则表示该描述子仅取这些 ID 的数据。可用于跨截面映射场景 |

与单点运算不同，截面运算的 `iid` 始终为全体截面 ID（而非分片后的部分 ID），因此不存在 `IDMode` 参数。

In [ ]:
from QuantStudio.Factor.FactorOperation import SectionOperator
print(qs_help(SectionOperator.calculate))

In [ ]:
# DTMode="单时点"：每时点分别调用
FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

@FactorOperatorized(operator_type="Section", args={"Name": "SectionFun", "Arity": 1, "DataType": "double", "DTMode": "单时点"})
def SectionFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")  # 全截面 ID
    print(f"x[0] : {x[0]}")
    print("-" * 10)
    return x[0] - np.nanmean(x[0])

SectionFactor = SectionFun(Close)

SectionIDs = ["000001.SZ", "000002.SZ", "000003.SZ"]
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 8), end_dt=dt.datetime(2025, 1, 10))
print(SectionFactor.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs, section_ids=SectionIDs))

In [ ]:
# DTMode="多时点"：所有时点一次调用，效率更高
@FactorOperatorized(operator_type="Section", args={"Name": "SectionFun", "Arity": 1, "DataType": "double", "DTMode": "多时点"})
def SectionFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")
    print(f"x[0] : {x[0]}")
    print("-" * 10)
    return x[0] - np.nanmean(x[0], axis=1, keepdims=True)

SectionFactor = SectionFun(Close)
print(SectionFactor.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs, section_ids=SectionIDs))

# 面板运算

同时依赖时间序列和整个截面数据的最复杂运算。兼具时序运算的回溯期参数和截面运算的截面范围参数。典型例子是时序截面双重标准化。

## 核心参数

`PanelOperator` 合并了 `TimeOperator` 和 `SectionOperator` 的参数：

| 参数 | 类型 | 说明 |
|------|------|------|
| `DTMode` | `"单时点"` / `"多时点"` | 每次调用处理的时点数量 |
| `LookBack` | `list[int]` | 每个描述子的回溯期数 |
| `StartDT` | `list[datetime \| None]` | 每个描述子的扩张窗口起始时点 |
| `iInitFactor` | `int` | 自身迭代因子索引 |
| `DescriptorSection` | `list[list[str] \| None]` | 每个描述子的截面范围 |

In [ ]:
from QuantStudio.Factor.FactorOperation import PanelOperator
print(qs_help(PanelOperator.calculate))

In [ ]:
# 面板运算：双重标准化
FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

@FactorOperatorized(operator_type="Panel", args={"Name": "PanelFun", "Arity": 1, "DataType": "double", "DTMode": "单时点", "LookBack": [3-1]})
def PanelFun(f, idt, iid, x, args):
    print(f"idt : {idt}")
    print(f"iid : {iid}")  # 全截面 ID
    print(f"x[0] : {x[0]}")  # shape=(LookBack+1, len(iid))
    print("-" * 10)
    Tmp = (x[0][-1] - np.nanmean(x[0], axis=0)) / np.nanstd(x[0], axis=0)
    return (Tmp - np.nanmean(Tmp)) / np.nanstd(Tmp)

PanelFactor = PanelFun(Close)

SectionIDs = ["000001.SZ", "000002.SZ", "000003.SZ"]
DTRuler = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 10))
DTs = DTRuler[-3:]
print(PanelFactor.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs, section_ids=SectionIDs, dt_ruler=DTRuler))

# 运算符重载

`Factor` 类重载了常用 Python 运算符，可以直接用表达式定义衍生因子。所有运算符重载在 `QuantStudio.Factor.BasicOperator` 中实现，本质上都是基于单点运算。

## 二元运算符

A、B 表示因子对象或具体标量数据：

| 表达式 | 含义 |
|--------|------|
| `A + B` | 对应相加 |
| `A - B` | 对应相减 |
| `A * B` | 对应相乘 |
| `A / B` | 对应相除 |
| `A // B` | 对应向下取整除法 |
| `A % B` | 对应取余 |
| `A ** B` | 对应取乘方 |
| `A < B` | 小于比较（返回 True/False） |
| `A <= B` | 小于等于比较 |
| `A > B` | 大于比较 |
| `A >= B` | 大于等于比较 |
| `A == B` | 相等比较 |
| `A != B` | 不等比较 |
| `A & B` | 逻辑与（要求 bool 类型） |
| `A \| B` | 逻辑或（要求 bool 类型） |
| `A ^ B` | 逻辑异或（要求 bool 类型） |

## 一元运算符

| 表达式 | 含义 |
|--------|------|
| `~A` | 取反操作 |
| `abs(A)` | 取绝对值 |
| `-A` | 取相反数 |
| `+A` | 正号（返回自身） |

另外，`BasicOperator` 提供了 `rename` 算子用于给因子对象重命名。

In [ ]:
# 运算符重载示例
from QuantStudio.Factor.BasicOperator import rename

FT = FDB.getTable(table_name="stock_cn_day_bar")
High, Low = FT.getFactor("high"), FT.getFactor("low")

Mid = rename((High + Low) / 2, factor_name="Mid")
print(Mid.Name)

DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 3))
print(Mid.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs))

# 内置算子

`QuantStudio.Factor.FactorOperator` 模块预定义了大量常用算子，可直接使用。所有内置算子均支持通过构造参数进行配置。

## 单点运算算子

| 算子 | 说明 | 关键构造参数 |
|------|------|-------------|
| `AsType` | 数据类型转换 | `dtype`：目标数据类型 |
| `Log` | 取对数 | `base`：底数（默认 e） |
| `NotNull` | 非 NULL 检测 | — |
| `IsIn` | 是否属于给定集合 | `test_elements`：检测集合 |
| `ApplyArrayFunc` | 施加对 array 整体运算的函数 | `func`：函数 |
| `Applymap` | 逐元素 map 操作 | `func`：函数 |
| `Where` | 条件选择（if-else） | — |
| `Fetch` | 从复合因子中取出子字段 | `pos`：位置或字段名 |
| `Sum` | 多因子求和 | `all_nan`：全为 NaN 时的替代值 |
| `Max` | 多因子求最大值 | `all_nan` |
| `Min` | 多因子求最小值 | `all_nan` |
| `Rank` | 排名 | `ascending`、`uniformization` |
| `Mean` | 多因子求平均值 | `weights`：权重列表 |
| `Std` | 多因子求标准差 | `ddof`：自由度 |
| `Regress` | 多因子线性回归取残差/系数 | `statistics`：输出的统计量类型 |
| `RegressChangeRate` | 回归系数变化率 | `LookBackNum` |
| `ToList` | 将多因子合并为 list | — |
| `ToCompound` | 将多因子合并为复合因子 | `compound_type` |

## 时序运算算子

| 算子 | 说明 | 关键构造参数 |
|------|------|-------------|
| `Lag` | 滞后算子 | `LagPeriod`：滞后期数 |
| `RollingRank` | 滚动排名 | `LookBack`、`ascending` |
| `RollingMean` | 滚动平均 | `LookBack`、`weights` |
| `RollingApply` | 通用滚动窗口函数 | `LookBack`、`func` |
| `RollingChangeRate` | 滚动变化率 | `LookBack`、`LookBackNum` |
| `RollingRegress` | 滚动回归 | `LookBack`、`statistics` |

## 截面运算算子

| 算子 | 说明 | 关键构造参数 |
|------|------|-------------|
| `SectionRank` | 截面排名 | `ascending`、`uniformization` |
| `Aggregate` | 截面聚合（分组映射） | `section_ids`、`func` |
| `Disaggregate` | 截面反聚合（从分组回映射到个股） | `section_ids` |
| `ConcatSection` | 截面拼接 | `section_ids` |
| `ChgSection` | 截面变化（与上一截面比较） | — |
| `SectionRegress` | 截面回归 | `statistics` |

## 面板运算算子

| 算子 | 说明 | 关键构造参数 |
|------|------|-------------|
| `PanelRegress` | 面板回归 | `LookBack`、`statistics` |

In [ ]:
# 内置算子示例：Log
from QuantStudio.Factor.FactorOperator import Log

log = Log(base=np.e)
print(qs_help(log))

FT = FDB.getTable(table_name="stock_cn_day_bar")
Close = FT.getFactor("close")

F = log(Close)
DTs = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 3))
print("-" * 10)
print(F.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs))

In [ ]:
# 内置算子示例：RollingRank (时序滚动排名)
from QuantStudio.Factor.FactorOperator import RollingRank

rr = RollingRank(LookBack=3, ascending=False)  # 3日滚动降序排名
Close = FDB.getTable("stock_cn_day_bar").getFactor("close")

F = rr(Close)

DTRuler = FT.getDateTime(start_dt=dt.datetime(2025, 1, 1), end_dt=dt.datetime(2025, 1, 10))
DTs = DTRuler[-5:]
print(F.readData(ids=["000001.SZ", "000002.SZ", "000003.SZ"], dts=DTs, dt_ruler=DTRuler))

# 数据因子

`DataFactor` 是直接赋予字面量数据（标量、DataFrame、Series）的因子，主要用于测试场景。它不需要依赖数据库连接，构造极为简便。

In [ ]:
from QuantStudio.Factor.Factor import DataFactor

F = DataFactor(data=1)
print(qs_help(F))

In [ ]:
# DataFactor 的 readData：标量会被广播到所有 ID 和时点
IDs = ["000001.SZ", "600519.SH"]
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(3)]

F = DataFactor(data=1)
print(F.readData(ids=IDs, dts=DTs))

print("-" * 10)
# 使用 DataFrame 构造：只有 DataFrame 中有值的时点/ID 才返回数据
F = DataFactor(data=pd.DataFrame(np.random.randn(2, 2), index=DTs[:2], columns=IDs))
print(F.readData(ids=IDs, dts=DTs))